# Stylometric SLM — one-click Colab training

Fine-tunes `google/mt5-base` on multilingual authorship attribution (14 authors, 4 languages).

## Run order (read this — it saves time)

1. **Set HF_TOKEN**: left sidebar → key icon → add `HF_TOKEN` with your Hugging Face token (free at https://huggingface.co/settings/tokens — needs read+write scope)
2. **Edit cell #1**: change `HF_USERNAME = "your-username"` to your actual HF username
3. **Upload the dataset BEFORE running anything**:
   - Click the folder icon (left sidebar) → upload icon → select `splits/dataset.jsonl` from your machine
   - It lands at `/content/dataset.jsonl`
   - Stage 0 will detect it and move it to the right place
4. **Runtime → Change runtime type → T4 GPU**
5. **Runtime → Run all**
6. Wait ~45–60 min, download `results.json` from the Files panel

If Stage 0 can't find the file, it stops with a clear error and tells you exactly what to upload.

In [ ]:
# === Config ===
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, "HF_TOKEN missing — add it in Colab secrets panel (left sidebar, key icon)"

# CHANGE THIS to your actual HF username
HF_USERNAME = "your-username"
HF_DATASET_REPO = f"{HF_USERNAME}/stylometric-slm-corpus"
HF_MODEL_REPO = f"{HF_USERNAME}/stylometric-slm-mt5"

BASE_MODEL = "google/mt5-base"
EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACCUM = 4
LR = 3e-5
MAX_INPUT_LEN = 1024
MAX_TARGET_LEN = 32
PREFIX = "classify authorship: "

os.environ['HF_TOKEN'] = HF_TOKEN
print(f"config loaded — uploading to /{HF_USERNAME}/...")

In [ ]:
# === Stage 0: Locate dataset.jsonl ===
# The notebook needs /content/splits/dataset.jsonl (17MB, ~2,987 rows).
# We check FOUR possible locations in order. First match wins.
#
# To get the file there: download splits/dataset.jsonl from your local machine
# (path: C:/Users/alaga/ghwork/stylometric-slm/splits/dataset.jsonl).
# Then click the folder icon (left sidebar) → upload icon → select dataset.jsonl.
# It lands at /content/dataset.jsonl. This cell handles the rest.

import shutil
from pathlib import Path

TARGET = Path("/content/splits/dataset.jsonl")
TARGET.parent.mkdir(parents=True, exist_ok=True)

candidates = [
    Path("/content/splits/dataset.jsonl"),    # already in place
    Path("/content/dataset.jsonl"),            # uploaded via Files panel
    Path("/content/splits/dataset.json"),     # alt extension
    Path("/content/dataset.json"),             # alt name + extension
    Path("/content/data.jsonl"),
]

found = None
for c in candidates:
    if c.exists() and c.stat().st_size > 1000:
        found = c
        break

if found is None:
    # Last-resort: try wget from GitHub (works only if you've pushed the repo)
    print("Not found in expected locations. Trying GitHub download...")
    GH_USER = "Rawbeew"  # change to your GitHub username if you push the repo
    BRANCH = "main"
    url = f"https://raw.githubusercontent.com/{GH_USER}/stylometric-slm/{BRANCH}/splits/dataset.jsonl"
    !wget -q "$url" -O /content/splits/dataset.jsonl
    if TARGET.exists() and TARGET.stat().st_size > 1000:
        found = TARGET
        print(f"Downloaded from {url}")

if found is None:
    print()
    print("=" * 60)
    print("DATASET NOT FOUND")
    print("=" * 60)
    print()
    print("Upload dataset.jsonl using ONE of these methods:")
    print()
    print("Method 1 — Files panel (recommended):")
    print("  1. Click the folder icon in the left sidebar")
    print("  2. Click the upload icon (paper-with-arrow) at the top")
    print("  3. Select splits/dataset.jsonl from your machine")
    print("  4. Re-run this cell")
    print()
    print("Method 2 — GitHub (if you've pushed the repo):")
    print(f"  Edit GH_USER in this cell to your GitHub username")
    print(f"  Edit BRANCH to your default branch (main or master)")
    print(f"  Re-run this cell")
    print()
    raise FileNotFoundError(
        "dataset.jsonl not found. See instructions above. "
        "The file is on your local machine at: "
        "C:/Users/alaga/ghwork/stylometric-slm/splits/dataset.jsonl"
    )

if found != TARGET:
    shutil.copy(found, TARGET)
    print(f"Found {found} ({found.stat().st_size:,} bytes), copied to {TARGET}")
else:
    print(f"Dataset already at {TARGET} ({TARGET.stat().st_size:,} bytes)")

# Final sanity check
assert TARGET.exists(), f"{TARGET} still missing"
size = TARGET.stat().st_size
assert size > 1_000_000, f"{TARGET} too small ({size} bytes) — file may be corrupt"
print(f"OK: {TARGET} ready, {size:,} bytes")

In [ ]:
# === Stage 1: Upload dataset.jsonl to your HF Hub ===
from huggingface_hub import HfApi, create_repo, upload_file
from pathlib import Path

ds_path = Path("/content/splits/dataset.jsonl")
assert ds_path.exists() and ds_path.stat().st_size > 1_000_000, \
    f"dataset.jsonl missing or too small at {ds_path}"

api = HfApi(token=HF_TOKEN)
create_repo(HF_DATASET_REPO, repo_type="dataset", private=False, token=HF_TOKEN, exist_ok=True)
print(f"repo ready: https://huggingface.co/datasets/{HF_DATASET_REPO}")

upload_file(
    path_or_fileobj=str(ds_path),
    path_in_repo="dataset.jsonl",
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
)
print(f"dataset.jsonl uploaded ({ds_path.stat().st_size:,} bytes)")

In [ ]:
# === Stage 2: Install dependencies ===
!pip install -q "transformers>=4.45" "datasets>=2.20" "sentencepiece>=0.2" accelerate
print("installed")

In [ ]:
# === Stage 3: Load dataset from Hub ===
from datasets import load_dataset

ds = load_dataset(HF_DATASET_REPO, token=HF_TOKEN)
print(f"loaded from Hub")
print(f"splits: {list(ds.keys())}")
print(f"total rows: {sum(len(ds[s]) for s in ds)}")

train_ds_raw = ds.filter(lambda x: x["split"] == "train")
eval_ds_raw = ds.filter(lambda x: x["split"] == "test")
print(f"train: {len(train_ds_raw)}, eval: {len(eval_ds_raw)}")
print(f"languages: {sorted(set(train_ds_raw['language']))}")
print(f"authors: {sorted(set(train_ds_raw['author']))}")

In [ ]:
# === Stage 4: Load mT5-base ===
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if not torch.cuda.is_available():
    print("WARNING: no GPU detected. Training will be very slow or may OOM.")
    print("Go to Runtime → Change runtime type → T4 GPU, then re-run.")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, token=HF_TOKEN)
print(f"loaded {BASE_MODEL}")
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# === Stage 5: Preprocess — text-to-text format ===
def preprocess(batch):
    inputs = [PREFIX + t for t in batch["text"]]
    targets = batch["author"]
    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LEN,
        truncation=True, padding="max_length",
    )
    labels = tokenizer(
        targets, max_length=MAX_TARGET_LEN,
        truncation=True, padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = train_ds_raw.map(preprocess, batched=True, remove_columns=train_ds_raw.column_names)
eval_ds = eval_ds_raw.map(preprocess, batched=True, remove_columns=eval_ds_raw.column_names)
print(f"preprocessed: train={len(train_ds)}, eval={len(eval_ds)}")

In [ ]:
# === Stage 6: Train ===
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = TrainingArguments(
    output_dir="/content/mt5-stylometric",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.05,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="epoch",
    bf16=torch.cuda.is_available(),
    push_to_hub=True,
    hub_model_id=HF_MODEL_REPO,
    hub_token=HF_TOKEN,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print(f"starting training: {EPOCHS} epochs, batch={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}, lr={LR}")
trainer.train()

In [ ]:
# === Stage 7: Push final model to Hub ===
trainer.push_to_hub(commit_message="mt5-base fine-tuned on multilingual authorship attribution (14 authors, 4 languages)")
print(f"pushed: https://huggingface.co/{HF_MODEL_REPO}")

In [ ]:
# === Stage 8: Per-author per-language accuracy ===
import json
from collections import defaultdict

correct = defaultdict(lambda: defaultdict(int))
total = defaultdict(lambda: defaultdict(int))

model.eval()
with torch.no_grad():
    for example in eval_ds_raw:
        inp = tokenizer(
            PREFIX + example["text"],
            return_tensors="pt",
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(model.device)
        out = model.generate(**inp, max_length=MAX_TARGET_LEN)
        pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        gold = example["author"]
        lang = example["language"]
        total[lang][gold] += 1
        if pred == gold:
            correct[lang][gold] += 1

print("accuracy per author per language:")
results = {}
for lang in sorted(total.keys()):
    results[lang] = {}
    for author in sorted(total[lang].keys()):
        c = correct[lang][author]
        t = total[lang][author]
        acc = c / t if t else 0.0
        results[lang][author] = {"correct": c, "total": t, "accuracy": acc}
        print(f"  {lang}/{author:<22s} {acc:.3f}  ({c}/{t})")
    overall_c = sum(correct[lang].values())
    overall_t = sum(total[lang].values())
    print(f"  {lang}/<overall>               {overall_c/overall_t:.3f}  ({overall_c}/{overall_t})")
    print()

with open("/content/results.json", "w") as f:
    json.dump(results, f, indent=2)
print("results saved to /content/results.json — download from Files panel")